# Benchmarking data

This notebook generates the data used in the benchmarking suite. 

In particular, the `_metadata` file can take a little while to generate, so we don't want to do it for every benchmarking run.

In [ ]:
import numpy as np

from hats.catalog import PartitionInfo, TableProperties
from hats.pixel_math import HealpixPixel
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from hats.io.file_io.file_pointer import get_upath
import tempfile
from hats.io.parquet_metadata import write_parquet_metadata
from hats.io.paths import new_pixel_catalog_file
from hats.pixel_math.spatial_index import healpix_to_spatial_index

In [ ]:
def write_to_metadata_files(partition_info: PartitionInfo, catalog_base_dir, **kwargs):
    with tempfile.TemporaryDirectory() as temp_pq_file:
        temp_dataset_dir = get_upath(temp_pq_file)
        for pixel in partition_info.get_healpix_pixels():
            batch = pa.RecordBatch.from_arrays(
                [[healpix_to_spatial_index(pixel.order, pixel.pixel)]],
                names=["_healpix_29"],
            )
            temp_info_table = pa.Table.from_batches([batch])
            pq.write_table(temp_info_table, new_pixel_catalog_file(temp_dataset_dir, pixel))
        return write_parquet_metadata(
            temp_pq_file, output_path=catalog_base_dir, create_per_partition_stats=True, **kwargs
        )

## Large catalog

This contains 196_607 partitions at order 7. This might seem like a silly number, and I guess it is, but it keeps the `_metadata` file under the github size limit.

In [ ]:
pixel_list = [HealpixPixel(7, pixel) for pixel in np.arange(196_608)]
partition_info = PartitionInfo.from_healpix(pixel_list)

catalog_base_dir = Path("large_catalog")
catalog_base_dir.mkdir(exist_ok=True)

partition_info.write_to_file(catalog_base_dir / "partition_info.csv")
write_to_metadata_files(partition_info, catalog_base_dir)

table_properties = TableProperties(
    catalog_name="large_catalog",
    catalog_type="object",
    total_rows=196_608,
    ra_column="",
    dec_column="",
)
table_properties.to_properties_file(catalog_base_dir)

## Midsize catalog

This contains 30_000 partitions at order 6.

In [ ]:
pixel_list = [HealpixPixel(6, pixel) for pixel in np.arange(30_000)]
partition_info = PartitionInfo.from_healpix(pixel_list)

catalog_base_dir = Path("midsize_catalog")
catalog_base_dir.mkdir(exist_ok=True)

partition_info.write_to_file(catalog_base_dir / "partition_info.csv")
write_to_metadata_files(partition_info, catalog_base_dir)

table_properties = TableProperties(
    catalog_name="midsize_catalog",
    catalog_type="object",
    total_rows=30_000,
    ra_column="",
    dec_column="",
)
table_properties.to_properties_file(catalog_base_dir)